# Custom transformer examples

Small synthetic examples for Arrow distance computation and fitted-statistic persistence.
Batch history contracts are documented in `docs/feature_dictionary.md`.


In [ ]:
import os
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "custom_transformers.py").is_file())
sys.path.insert(0, str(root))
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession, functions as F
from custom_transformers import haversine_km, OutlierIQRTruncator, TargetEncoder
spark = SparkSession.builder.master("local[2]").appName("transformer-examples").getOrCreate()


## Arrow Haversine

The same point has zero distance. Invalid coordinates produce null/NaN.


In [ ]:
points = spark.createDataFrame([(10.0, 20.0, 10.0, 20.0)], ["lat1", "lon1", "lat2", "lon2"])
points.select(haversine_km(*[F.col(c) for c in points.columns]).alias("distance_km")).show()


## Train-only statistics

Fit bounds and category target means only on training data. Unseen categories use the training global mean. These examples do not measure production model quality.


In [ ]:
train = spark.createDataFrame([("A", 1.0), ("A", 2.0), ("B", 3.0), ("B", 100.0)], ["category", "value"])
iqr = OutlierIQRTruncator(inputCols=["value"], outputCols=["value_clipped"]).fit(train)
iqr.transform(train).show()
encoder = TargetEncoder(inputCols=["category"], outputCols=["encoded"], targetCol="value").fit(train)
encoder.transform(spark.createDataFrame([("C",)], ["category"])).show()
spark.stop()
